# Result

In [ ]:
# Data for comparison
target_date = test_org['Date'].iloc[TIME_STEPS:] 
target_VIX = test_org['vix_ln_return'].iloc[TIME_STEPS:]
predictions_hv = test_org['Historical Volatility'].iloc[TIME_STEPS:]

# Function to scale back predictions
def scale_back(predictions):
    predictions = [prediction[0] for prediction in predictions] # unpack array

    # scale back
    mean_vol = train_org['vix'].mean()
    std_vol = train_org['vix'].std() 
    predictions = [prediction * std_vol + mean_vol for prediction in predictions]
    
    return predictions

# Predict
predictions_nn = scale_back(model_nn.predict(x_test))
predictions_lstm = scale_back(model_lstm.predict(x_test))
predictions_gru = scale_back(model_gru.predict(x_test))

# pd.Series-ization
predictions_nn = pd.Series(predictions_nn, index=target_VIX.index, name='predictions_nn')
predictions_lstm = pd.Series(predictions_lstm, index=target_VIX.index, name='predictions_lstm')
predictions_gru = pd.Series(predictions_gru, index=target_VIX.index, name='predictions_gru')

In [ ]:
label_date = train_org['Date']#.iloc[TIME_STEPS:] 
label_VIX = train_org['vix']#.iloc[:TIME_STEPS]

In [ ]:
# Plot
fig1, ax1 = plt.figure(figsize=(20,7)), plt.axes()
fig2, ax2 = plt.figure(figsize=(20,7)), plt.axes()
fig3, ax3 = plt.figure(figsize=(20,7)), plt.axes()
fig4, ax4 = plt.figure(figsize=(20,7)), plt.axes()

# label_date 
ax1.plot(label_date, label_VIX, label='VIX (train data)')
ax2.plot(label_date, label_VIX, label='VIX (train data)')
ax3.plot(label_date, label_VIX, label='VIX (train data)')
ax4.plot(label_date, label_VIX, label='VIX (train data)')

# target data
ax1.plot(target_date, target_VIX, label='VIX (target data)')
ax2.plot(target_date, target_VIX, label='VIX (target data)')
ax3.plot(target_date, target_VIX, label='VIX (target data)')
ax4.plot(target_date, target_VIX, label='VIX (target data)')

# predictions
ax1.plot(target_date, predictions_hv, label='GARCH',color=palette[1])
ax2.plot(target_date, predictions_nn, label='GARCH-NN',color=palette[2])
ax3.plot(target_date, predictions_lstm, label='GARCH-LSTM',color=palette[3])
ax4.plot(target_date, predictions_gru, label='GARCH-GRU',color=palette[4])

# show
ax1.legend()
ax2.legend()
ax3.legend()
ax4.legend()
plt.show()

In [ ]:
# Plot
fig1, ax1 = plt.figure(figsize=(20,7)), plt.axes()
fig2, ax2 = plt.figure(figsize=(20,7)), plt.axes()
fig3, ax3 = plt.figure(figsize=(20,7)), plt.axes()
fig4, ax4 = plt.figure(figsize=(20,7)), plt.axes()

# target data
ax1.plot(target_date, target_VIX, label='VIX (target data)')
ax2.plot(target_date, target_VIX, label='VIX (target data)')
ax3.plot(target_date, target_VIX, label='VIX (target data)')
ax4.plot(target_date, target_VIX, label='VIX (target data)')

# predictions
ax1.plot(target_date, predictions_hv, label='GARCH',color=palette[1])
ax2.plot(target_date, predictions_nn, label='GARCH-NN',color=palette[2])
ax3.plot(target_date, predictions_lstm, label='GARCH-LSTM',color=palette[3])
ax4.plot(target_date, predictions_gru, label='GARCH-GRU',color=palette[4])

# show
ax1.legend()
ax2.legend()
ax3.legend()
ax4.legend()
plt.show()

# MSE and MAE

In [ ]:
# Calculate error
def cal_MSE(predictions):
    errors = (predictions - target_VIX)**2
    return errors.mean()

def cal_MAE(predictions):
    errors = abs(predictions - target_VIX)
    return errors.mean()


# Print error
print('GARCH')
print('MSE:',cal_MSE(predictions_hv))
print('MAE:',cal_MAE(predictions_hv))

print('GARCH-NN')
print('MSE:',cal_MSE(predictions_nn))
print('MAE:',cal_MAE(predictions_nn))

print('GARCH-LSTM')
print('MSE:',cal_MSE(predictions_lstm))
print('MAE:',cal_MAE(predictions_lstm))

print('GARCH-GRU')
print('MSE:',cal_MSE(predictions_gru))
print('MAE:',cal_MAE(predictions_gru))

# Number of best predictions

In [ ]:
# Compare day by day

# error list
errors_hv = abs(target_VIX - predictions_hv)
errors_nn = abs(target_VIX - predictions_nn)
errors_lstm = abs(target_VIX - predictions_lstm)
errors_gru = abs(target_VIX - predictions_gru)

# count the best models
best_models = []
for i in range(len(errors_hv)):
    each_predictions = [errors_hv.iloc[i], errors_nn.iloc[i], errors_lstm.iloc[i], errors_gru.iloc[i]]
    best_models.append(each_predictions.index(min(each_predictions)))

best_counts = [best_models.count(i) for i in range(4)]

    
# plot
fig, ax = plt.figure(figsize=(10,7)), plt.axes()
ax.pie(best_counts, labels=['GARCH','GARCH-NN','GARCH-LSTM','GARCH-GRU'], autopct='%.0f%%', startangle=90, colors=palette[1:])
plt.title('Number of days in which it performed the best')
plt.show()

# Number of predicting the right direction (up or down)

In [ ]:
# direction arrays
directions_VIX = target_VIX - target_VIX.shift()
directions_hv = predictions_hv - predictions_hv.shift()
directions_nn = predictions_nn - predictions_nn.shift()
directions_lstm = predictions_lstm - predictions_lstm.shift()
directions_gru = predictions_gru - predictions_gru.shift()

# multiplicate element-wise, and count the positivie elements
def count_pos(series):
    count = 0
    for x in series:
        if x>0:
            count += 1
    return count

direction_counts = [62,0,0,0,0]
direction_counts[1] = count_pos(directions_hv * directions_VIX)
direction_counts[2] = count_pos(directions_nn * directions_VIX)
direction_counts[3] = count_pos(directions_lstm * directions_VIX)
direction_counts[4] = count_pos(directions_gru * directions_VIX)

# plot
fig, ax = plt.figure(figsize=(10,7)), plt.axes()
ax.bar(['Total Days','GARCH','GARCH-NN','GARCH-LSTM','GARCH-GRU'],direction_counts, color=palette)
plt.title('Number of days in which it predicted the right direction(up or down)')
plt.show()

# direction prediction accuracy
print('GARCH')
print(direction_counts[1] / direction_counts[0]*100,'%')
print('GARCH-NN')
print(direction_counts[2] / direction_counts[0]*100,'%')
print('GARCH-LSTM')
print(direction_counts[3] / direction_counts[0]*100,'%')
print('GARCH-GRU')
print(direction_counts[4] / direction_counts[0]*100,'%')